# **Fairness-Aware Age Estimation: Bias Detection and Mitigation - Part II**
UB Master in Fundamental Principels of Data Science (2025-2026)

Author: Julio C. S. Jacques Junior

Last modified: Jan, 2026.

---

# **Part II: Goal**

- Use **data augmentation** to address the bias problem.

- In this notebook, we apply data augmentation to all samples with age > 60, and to all other samples with a probability of 25%. **You are expected to define a more creative solution**, in which other attributes such as gender, ethnicity, facial expression, or different age ranges are also considered.

- You may also use external datasets to build a more balanced training set, or generate synthetic samples using a generative model (e.g., [Stable Diffusion](https://huggingface.co/blog/stable_diffusion)).

- **Requirements:** Carefully review the instructions in Notebook **Part I** and run it.


## Checking the pytorch version
 - This notebook was successfully tested on version = 2.9.0+cu126

In [ ]:
import torch
print(torch.__version__)

## Importing required libraries

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import csv
from PIL import Image
from timm import create_model
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
from tqdm import tqdm

# **Downloading the Appa-Real Dataset**
- Please, check the **detailed instructions in notebook Part I**.

In [ ]:
from zipfile import ZipFile

# downloading the data
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2025/appa-real-dataset_v2.zip

with ZipFile('appa-real-dataset_v2.zip','r') as zip:
   zip.extractall()
   print('Data decompressed successfully')

# removing the .zip file after extraction to clean space
!rm appa-real-dataset_v2.zip

# **Mount Google Drive to save the trained model on the cloud**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
# Note, the default path will be: '/content/gdrive/MyDrive/'
# In my case, the final path will be: '/content/gdrive/MyDrive/temp/' as I
# created a '/temp/' folder in my google drive for this purpose.

# **Defining the Data Loader Class**
- In this example, metadata information is loaded but not used. Future implementations can take benefit of it.
- Note that age labels are divided by 100 (assuming 100 is the max age found in the dataset) so that the age values can be normalized to be in the range of 0 and 1. This way, we can add a sigmoid activation in the last layer of our model.
- The method `__getitem__` implements a **simple data augmentation strategy based only on the age label**: all training samples with age > 60 are transformed using the transformations defined in `augmented_transforms`. All other training samples are transformed using the same transformations, but with a probability of 25%.


In [ ]:
class AgeEstimationDataset(Dataset):
    def __init__(self, image_dir, csv_file, base_transforms, augmented_transforms=None, augmentation_prob=None):
        self.image_dir = image_dir
        self.data_info = pd.read_csv(csv_file)
        self.base_transforms = base_transforms
        self.augmented_transforms = augmented_transforms
        self.age_normalization_factor = 100.0  # used to normalize age labels
        self.augmentation_prob = augmentation_prob  # probability for random augmentation

    def __len__(self):
        return len(self.data_info)

    def __normalization_factor__(self):
        return self.age_normalization_factor

    def __getitem__(self, idx):
        # Load image
        image_id = f"{int(self.data_info.iloc[idx, 0]):06d}.jpg"
        image_path = os.path.join(self.image_dir, image_id)
        image = Image.open(image_path).convert("RGB")

        # Load age and normalize
        raw_age = float(self.data_info.iloc[idx, 1])
        age = raw_age / self.age_normalization_factor

        metadata = self.data_info.iloc[idx, 2:].tolist()  # Optional metadata

        # ----------------------------
        # TRANSFORM LOGIC
        # ----------------------------
        if self.augmented_transforms is not None:
          # Training behavior (augmentation enabled)
          # Conditional augmentation
          if raw_age >= 60:
              # Always apply augmentation for age >= 60
              image = self.augmented_transforms(image)
          else:
              # Apply augmentation with probability
              if random.random() < self.augmentation_prob:
                  image = self.augmented_transforms(image)
              else:
                  image = self.base_transforms(image)
        else:
          # Validation / Test behavior (no augmentation)
          image = self.base_transforms(image)

        return image, torch.tensor(age, dtype=torch.float32), metadata

# **Defining the base image transformations**

In [ ]:
base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# **Defining the transformations used for data augmentation**

In [ ]:
augmented_transforms = transforms.Compose([
    # --- Geometry ---
    transforms.RandomResizedCrop(
        224, scale=(0.85, 1.0), ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10, fill=0),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        shear=5,
        fill=0
    ),

    # --- Photometric ---
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
        hue=0.02
    ),

    transforms.RandomGrayscale(p=0.1),

    # --- Convert to Tensor FIRST ---
    transforms.ToTensor(),

    # --- Occlusion (Tensor-only) ---
    transforms.RandomErasing(
        p=0.3,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value=0
    ),

    # --- Normalize ---
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

# **Loading the Train and Validation sets**

In [ ]:
# Create dataset and dataloader (train set):

# train set with data augmentation
dataset_train = AgeEstimationDataset("train_data", "labels_metadata_train.csv", base_transforms=base_transforms, augmented_transforms=augmented_transforms, augmentation_prob=0.25)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of train samples: {len(dataloader_train.dataset)}")

# Create dataset and dataloader (validation set):
dataset_valid = AgeEstimationDataset("valid_data", "labels_metadata_valid.csv", base_transforms=base_transforms)
dataloader_valid = DataLoader(dataset_valid, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of valid samples: {len(dataloader_valid.dataset)}")

# **Visualizing Some Augmented Data Samples**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_augmented_samples(dataset, indices=None):
    """
    Displays original vs augmented versions for 3 samples.

    Args:
        dataset: Your PyTorch dataset
        indices: Optional list of 3 indices. If None, uses first 3 samples.
    """

    if indices is None:
        indices = [0, 1, 2]

    fig, axes = plt.subplots(len(indices), 2, figsize=(8, 4 * len(indices)))

    for i, idx in enumerate(indices):
        # Load raw image without transform
        image_id = f"{dataset.data_info.iloc[idx, 0]:06d}.jpg"
        image_path = os.path.join(dataset.image_dir, image_id)
        image = Image.open(image_path).convert("RGB")

        raw_age = float(dataset.data_info.iloc[idx, 1])

        # Apply base and augmented transforms
        base_img = dataset.base_transforms(image)
        aug_img = dataset.augmented_transforms(image)

        # Convert tensors to numpy for plotting
        base_img = base_img.permute(1, 2, 0).numpy()
        aug_img = aug_img.permute(1, 2, 0).numpy()

        # Undo normalization if needed (assuming mean=0.5, std=0.5)
        base_img = (base_img * 0.5) + 0.5
        aug_img = (aug_img * 0.5) + 0.5

        axes[i, 0].imshow(base_img)
        axes[i, 0].set_title(f"Original (Age: {raw_age})")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(aug_img)
        axes[i, 1].set_title("Augmented")
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_augmented_samples(dataset_train, indices=[10, 30, 80])

# **Loading the pretrained ViT baselone model and adapting it to our problem**

- ViT normally outputs class scores for classification tasks; here, we adapt it for regression by setting **num_classes=1**.

> **Note:** This notebook is intended as a starting point. For your deliverables, avoid making only minor modifications. Instead, explore your creativity and try more substantial improvements.

- **It is also recommended to use the same architecture when comparing results with and without data augmentation, in order to ensure a fair comparison.**


In [ ]:
# Vision Transformer Model for Age Prediction (pretrained on ImageNet)
# https://pytorch.org/vision/main/models/vision_transformer.html
# https://huggingface.co/docs/transformers/main/en//model_doc/vit
class AgeEstimationViT(nn.Module):
    def __init__(self):
        super(AgeEstimationViT, self).__init__()
        self.vit = create_model("vit_base_patch16_224", pretrained=True, num_classes=1) # num_classes=1 as we want to regress a single (age value)
        self.activation = nn.Sigmoid()  # Added Sigmoid activation

    def forward(self, x):
        x = self.vit(x)
        return self.activation(x)  # Apply Sigmoid activation

In [ ]:
# creating the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeEstimationViT().to(device)

# Defining an auxiliary function to plot the training history

In [ ]:
# Function to plot training curves
def plot_training_curves(train_losses, val_losses):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.legend()
    plt.grid()
    plt.show()

# **Defining the Training function**
- Our training code includes **early stopping** and **automatic saving of the best model**. Early stopping monitors the validation loss during training and halts training if the model stops improving for a specified number of epochs, helping to prevent overfitting and save computational resources. At the same time, the model with the lowest validation loss is automatically saved, ensuring that we keep the best-performing version for evaluation or deployment.

In [ ]:
import random

# Training function with early stopping and model saving
def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs, patience, model_path):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    early_stopping_counter = 0
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0

            with tqdm(dataloaders[phase], desc=f"{phase.capitalize()} Epoch {epoch+1}") as t:
                for inputs, labels, _ in t:
                    inputs, labels = inputs.to(device), labels.to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        loss = criterion(outputs.view_as(labels), labels)

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    t.set_postfix(loss=loss.item())

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            print(f'{phase} Loss: {epoch_loss:.6f}')

            if phase == 'train':
                train_losses.append(epoch_loss)
            else:
                val_losses.append(epoch_loss)
                scheduler.step(epoch_loss)

                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    early_stopping_counter = 0
                    # Save best model during training
                    print("saving best model...")
                    torch.save(best_model_wts, model_path)
                else:
                    early_stopping_counter += 1
                    if early_stopping_counter >= patience:
                        print("Early stopping triggered.")
                        model.load_state_dict(best_model_wts)
                        plot_training_curves(train_losses, val_losses)
                        return model

    model.load_state_dict(best_model_wts)
    plot_training_curves(train_losses, val_losses)
    return model

# **Training the Model (now with data augmentation)**

- In the next cell, **we train our model using the same architecture and hyperparameters as in Part I**.
- The only difference is that **we now apply data augmentation**, as described above.



In [ ]:
# model with data augmentation
model_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model_aug.pth"

# model hyperparameters
num_epochs = 50
patience = 10
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=patience)

# data loaders
dataloaders = {"train": dataloader_train, "val": dataloader_valid}  # Assuming split dataset

# train the model
best_model = train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs, patience, model_filename)

# **Defining an auxiliary function to evaluate the model**
- Check Part I for the details.


In [ ]:
# Function to make predictions on test set and compute MSE
def predict_and_evaluate(model_path, test_dataset, output_zip=None, batch_size=32, output_csv="predictions.csv"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AgeEstimationViT().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    predictions = []
    actual_ages = []

    with torch.no_grad():
        for images, labels, metadata in tqdm(test_loader, desc="Predicting"):
            images = images.to(device)
            outputs = model(images).squeeze().cpu().numpy()
            labels = labels.cpu().numpy()

            predictions.extend(outputs * test_dataset.__normalization_factor__())
            actual_ages.extend(labels * test_dataset.__normalization_factor__())

    mae = np.mean(np.abs(np.array(predictions) - np.array(actual_ages)))
    if output_zip is not None:
      print(f"\n=======\nMean Absolute Error on Test Set: {mae:.4f}")
    else:
      print(f"\n=======\nMean Absolute Error on Validation Set: {mae:.4f}")


    # Only create ZIP if output_zip is provided
    if output_zip is not None:
        # Save predictions to CSV without headers
        with open(output_csv, mode='w', newline='') as file:
            writer = csv.writer(file)
            for pred in predictions:
                writer.writerow([pred])

        with ZipFile(output_zip, 'w') as zipf:
            zipf.write(output_csv, os.path.basename(output_csv))
        print(f"Predictions saved to {output_csv} and compressed as {output_zip}")

    return predictions, mae

# **Loading the Saved Model and Making Predictions on the Validation Set**


In [ ]:
model_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model_aug.pth"

# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_valid,output_zip=None, batch_size=32,output_csv=None)



---



# **Generating the submission fie (on the test set) for our challenge**

- Loading the Saved Model and Making Predictions on the Test Set

- The following cells are generating predictions (and evaluating them) on the **Test set** so that we can create our submission file to be uploaded to our age estimation challenge.

- **Do not evaluate your model on the Test set when defining your model, training strategy, or hyperparameters.** For this, use the Validation set.

In [ ]:
# Create dataset and dataloader (test set):
dataset_test = AgeEstimationDataset("test_data", "labels_metadata_test.csv", base_transforms=base_transforms)
print(f"Total number of test samples: {len(dataset_test)}")

In [ ]:
# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_test, "predictionsViTbaseline_aug.zip")